# 600 — Program–Drug Associations

## Objective

Characterize lineage-aware associations between the frozen Phase 4 consensus programs and pharmacogenomic drug-response measurements across GDSC, CTRP, and PRISM.

This notebook treats pharmacogenomic results as computational associations describing resistance-like contexts. It does not establish clinical resistance, therapeutic efficacy, causal mechanisms, or validated biomarkers.

## Analytical boundary

Before any program–drug association is estimated, this notebook will:

1. validate the frozen upstream program and cell-model handoffs;
2. characterize identifier compatibility and model coverage across GDSC, CTRP, and PRISM;
3. characterize drug and screen coverage without using association results to select drugs, models, metrics, thresholds, or lineages;
4. determine whether external program scores can be obtained by direct overlap or by deterministic projection of frozen Phase 4 gene weights, without refitting, reorientation, or reweighting;
5. document the information required to freeze the prospective Phase 6 analysis contract.

No inferential program–drug results will be inspected before the applicable Phase 6 analytical rules are frozen.

## Evidence roles

- **GDSC:** developmental/internal pharmacogenomic characterization. GDSC contributed to upstream Phase 3 analyses and is therefore not treated as independent external replication.
- **CTRP:** external cross-screen pharmacogenomic resource.
- **PRISM:** external cross-screen pharmacogenomic resource, preserving screen and compound-batch structure until prospective aggregation rules are defined.

Cell-line overlap, lineage structure, drug identity, drug-family structure, screen structure, platform differences, and proliferation-related confounding will remain explicit throughout Phase 6.

Phase 5 functional-vulnerability results will not be used to select drugs, programs, models, thresholds, or hypotheses for this analysis.

In [1]:
# =============================================================================
# Imports
# =============================================================================

import zipfile

import numpy as np
import pandas as pd

from pancancer_epigenetics.utils.artifact_registry import (
    load_artifact_registry,
    resolve_artifact_path,
)
from pancancer_epigenetics.utils.paths import Paths
from pancancer_epigenetics.utils.raw_data_registry import load_raw_data_registry

In [2]:
# =============================================================================
# Load project registries
# =============================================================================

artifact_registry = load_artifact_registry()
raw_registry = load_raw_data_registry()

print(f"Frozen artifacts registered: {len(artifact_registry['artifacts'])}")
print(
    "Pharmacogenomic raw resources registered:",
    ", ".join(resource for resource in ("gdsc", "ctrp", "prism") if resource in raw_registry),
)

Frozen artifacts registered: 100
Pharmacogenomic raw resources registered: gdsc, ctrp, prism


In [3]:
# =============================================================================
# Resolve frozen upstream artifact paths
# =============================================================================

UPSTREAM_ARTIFACT_IDS = (
    "phase3.302.integrated_modeling_cohort",
    "phase3.303.harmonized_expression",
    "phase4.400.cross_system_shared_gene_universe",
    "phase4.401.consensus_transcriptomic_program_catalog",
    "phase4.401.consensus_cellline_scores",
    "phase4.401.consensus_transcriptomic_gene_weights",
)

upstream_artifact_paths = {
    artifact_id: resolve_artifact_path(artifact_registry, artifact_id)
    for artifact_id in UPSTREAM_ARTIFACT_IDS
}

for artifact_id in UPSTREAM_ARTIFACT_IDS:
    print(
        f"{artifact_id}: "
        f"{artifact_registry['artifacts'][artifact_id]['path']}"
    )

phase3.302.integrated_modeling_cohort: data/interim/metadata/302_integrated_modeling_cohort.csv
phase3.303.harmonized_expression: data/interim/expression/303_expression_harmonized.parquet
phase4.400.cross_system_shared_gene_universe: data/processed/consensus_programs/400_cross_system_shared_gene_universe.csv
phase4.401.consensus_transcriptomic_program_catalog: data/processed/consensus_programs/401_consensus_transcriptomic_program_catalog.csv
phase4.401.consensus_cellline_scores: data/processed/consensus_programs/401_consensus_cellline_scores.parquet
phase4.401.consensus_transcriptomic_gene_weights: data/processed/consensus_programs/401_consensus_transcriptomic_gene_weights.csv


In [4]:
# =============================================================================
# Validate frozen upstream handoffs
# =============================================================================

upstream_handoff_status = pd.DataFrame(
    [
        {
            "artifact_id": artifact_id,
            "status": artifact_registry["artifacts"][artifact_id]["status"],
            "exists": upstream_artifact_paths[artifact_id].is_file(),
        }
        for artifact_id in UPSTREAM_ARTIFACT_IDS
    ]
)

print(
    "All inputs frozen:",
    upstream_handoff_status["status"].eq("frozen").all(),
)
print(
    "All input files present:",
    upstream_handoff_status["exists"].all(),
)

upstream_handoff_status

All inputs frozen: True
All input files present: True


,artifact_id,status,exists
0,phase3.302.integrated_modeling_cohort,frozen,True
1,phase3.303.harmonized_expression,frozen,True
2,phase4.400.cross_system_shared_gene_universe,frozen,True
3,phase4.401.consensus_transcriptomic_program_ca...,frozen,True
4,phase4.401.consensus_cellline_scores,frozen,True
5,phase4.401.consensus_transcriptomic_gene_weights,frozen,True


In [5]:
# =============================================================================
# Load frozen cell-model cohort and consensus-program scores
# =============================================================================

modeling_cohort = pd.read_csv(
    upstream_artifact_paths["phase3.302.integrated_modeling_cohort"]
)
consensus_cellline_scores = pd.read_parquet(
    upstream_artifact_paths["phase4.401.consensus_cellline_scores"]
)

print("Modeling cohort shape:", modeling_cohort.shape)
print("Consensus cell-line scores shape:", consensus_cellline_scores.shape)

print("\nModeling cohort columns:")
print(modeling_cohort.columns.tolist())

print("\nConsensus score columns:")
print(consensus_cellline_scores.columns.tolist())

Modeling cohort shape: (713, 8)
Consensus cell-line scores shape: (713, 4)

Modeling cohort columns:
['ModelID', 'SangerModelID', 'COSMICID', 'CellLineName', 'OncotreeLineage', 'OncotreePrimaryDisease', 'OncotreeSubtype', 'CCLEName']

Consensus score columns:
['ModelID', 'CONSENSUS_TX_01', 'CONSENSUS_TX_02', 'CONSENSUS_TX_03']


In [6]:
# =============================================================================
# Validate cell-model identity across frozen handoffs
# =============================================================================

modeling_ids = set(modeling_cohort["ModelID"])
score_ids = set(consensus_cellline_scores["ModelID"])

print("Modeling cohort unique ModelID:", modeling_cohort["ModelID"].nunique())
print(
    "Consensus scores unique ModelID:",
    consensus_cellline_scores["ModelID"].nunique(),
)
print(
    "Duplicate ModelID in modeling cohort:",
    modeling_cohort["ModelID"].duplicated().sum(),
)
print(
    "Duplicate ModelID in consensus scores:",
    consensus_cellline_scores["ModelID"].duplicated().sum(),
)
print("ModelID sets identical:", modeling_ids == score_ids)
print("Models only in cohort:", len(modeling_ids - score_ids))
print("Models only in scores:", len(score_ids - modeling_ids))

Modeling cohort unique ModelID: 713
Consensus scores unique ModelID: 713
Duplicate ModelID in modeling cohort: 0
Duplicate ModelID in consensus scores: 0
ModelID sets identical: True
Models only in cohort: 0
Models only in scores: 0


In [7]:
# =============================================================================
# Assemble frozen Phase 6 anchor cohort
# =============================================================================

phase6_anchor_cohort = modeling_cohort.merge(
    consensus_cellline_scores,
    on="ModelID",
    how="inner",
    validate="one_to_one",
)

print("Phase 6 anchor cohort shape:", phase6_anchor_cohort.shape)
print(
    "Models retained:",
    phase6_anchor_cohort["ModelID"].nunique(),
)

phase6_anchor_cohort.head()

Phase 6 anchor cohort shape: (713, 11)
Models retained: 713


,ModelID,SangerModelID,COSMICID,CellLineName,OncotreeLineage,OncotreePrimaryDisease,OncotreeSubtype,CCLEName,CONSENSUS_TX_01,CONSENSUS_TX_02,CONSENSUS_TX_03
0,ACH-000001,SIDM00105,905933.0,NIH:OVCAR-3,Ovary/Fallopian Tube,Ovarian Epithelial Tumor,High-Grade Serous Ovarian Cancer,NIHOVCAR3_OVARY,-0.618908,0.121723,-0.047979
1,ACH-000002,SIDM00829,905938.0,HL-60,Myeloid,Acute Myeloid Leukemia,Acute Myeloid Leukemia,HL60_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,1.312391,0.062750,-0.407409
2,ACH-000004,SIDM00594,907053.0,HEL,Myeloid,Acute Myeloid Leukemia,Acute Myeloid Leukemia,HEL_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,1.236019,0.613855,-0.498685
3,ACH-000006,SIDM01023,908148.0,MONO-MAC-6,Myeloid,Acute Myeloid Leukemia,Acute Monoblastic/Monocytic Leukemia,MONOMAC6_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,1.496321,0.404348,-0.515095
4,ACH-000007,SIDM00677,907795.0,LS513,Bowel,Colorectal Adenocarcinoma,Colon Adenocarcinoma,LS513_LARGE_INTESTINE,-0.238521,-1.263317,-1.175949


In [8]:
# =============================================================================
# Characterize anchor-cohort lineage structure
# =============================================================================

lineage_counts = (
    phase6_anchor_cohort["OncotreeLineage"]
    .value_counts(dropna=False)
    .rename_axis("OncotreeLineage")
    .reset_index(name="n_models")
)

print("Number of lineages:", lineage_counts.shape[0])
print(
    "Models with missing lineage:",
    phase6_anchor_cohort["OncotreeLineage"].isna().sum(),
)
print("Smallest lineage size:", lineage_counts["n_models"].min())
print("Largest lineage size:", lineage_counts["n_models"].max())

lineage_counts

Number of lineages: 27
Models with missing lineage: 0
Smallest lineage size: 1
Largest lineage size: 136


,OncotreeLineage,n_models
0,Lung,136
1,Lymphoid,86
2,Esophagus/Stomach,51
3,Breast,47
4,Bowel,43
5,Skin,37
6,CNS/Brain,37
7,Ovary/Fallopian Tube,34
8,Myeloid,32
9,Pancreas,28


In [9]:
# =============================================================================
# Resolve Phase 6 pharmacogenomic input paths
# =============================================================================

PHARMACOGENOMIC_INPUT_ROLES = {
    "gdsc": {
        "response": "pharmacogenomic_response_matrix",
    },
    "ctrp": {
        "response_archive": "primary_pharmacogenomic_archive",
    },
    "prism": {
        "cell_line_metadata": "cell_line_metadata",
        "response": "dose_response_curve_parameters",
        "treatment_metadata": "replicate_collapsed_treatment_metadata",
    },
}

pharmacogenomic_input_paths = {}

for resource_id, inputs in PHARMACOGENOMIC_INPUT_ROLES.items():
    resource = raw_registry[resource_id]

    for input_name, role in inputs.items():
        matches = [
            file_name
            for file_name, metadata in resource["files"].items()
            if metadata.get("role") == role
        ]

        if len(matches) != 1:
            raise ValueError(
                f"{resource_id}: expected one file for role '{role}', "
                f"found {len(matches)}"
            )

        pharmacogenomic_input_paths[(resource_id, input_name)] = (
            Paths.root / resource["canonical_dir"] / matches[0]
        )

for key, path in pharmacogenomic_input_paths.items():
    print(f"{key}: {path.relative_to(Paths.root)}")

('gdsc', 'response'): data\raw\gdsc\GDSC2_fitted_dose_response_27Oct23.xlsx
('ctrp', 'response_archive'): data\raw\ctrp\CTRPv2.0_2015_ctd2_ExpandedDataset.zip
('prism', 'cell_line_metadata'): data\raw\prism\secondary-screen-cell-line-info.csv
('prism', 'response'): data\raw\prism\secondary-screen-dose-response-curve-parameters.csv
('prism', 'treatment_metadata'): data\raw\prism\secondary-screen-replicate-collapsed-treatment-info.csv


In [10]:
# =============================================================================
# Quantify direct-ID model coverage in GDSC and PRISM
# =============================================================================

gdsc_models = (
    pd.read_excel(
        pharmacogenomic_input_paths[("gdsc", "response")],
        usecols=["SANGER_MODEL_ID"],
    )
    .dropna()
    .drop_duplicates()
)

prism_models = (
    pd.read_csv(
        pharmacogenomic_input_paths[("prism", "response")],
        usecols=["depmap_id", "screen_id"],
    )
    .dropna(subset=["depmap_id"])
    .drop_duplicates()
)

anchor_sanger_ids = set(phase6_anchor_cohort["SangerModelID"].dropna())
anchor_model_ids = set(phase6_anchor_cohort["ModelID"])

gdsc_response_ids = set(gdsc_models["SANGER_MODEL_ID"])
gdsc_anchor_overlap = gdsc_response_ids & anchor_sanger_ids

prism_coverage_rows = []

for screen_id, screen_models in prism_models.groupby("screen_id"):
    response_ids = set(screen_models["depmap_id"])
    overlap_ids = response_ids & anchor_model_ids

    prism_coverage_rows.append(
        {
            "resource": f"PRISM {screen_id}",
            "response_models": len(response_ids),
            "anchor_overlap": len(overlap_ids),
            "anchor_coverage_pct": 100 * len(overlap_ids) / len(anchor_model_ids),
        }
    )

direct_id_coverage = pd.DataFrame(
    [
        {
            "resource": "GDSC",
            "response_models": len(gdsc_response_ids),
            "anchor_overlap": len(gdsc_anchor_overlap),
            "anchor_coverage_pct": 100 * len(gdsc_anchor_overlap) / len(anchor_sanger_ids),
        },
        *prism_coverage_rows,
    ]
)

direct_id_coverage

,resource,response_models,anchor_overlap,anchor_coverage_pct
0,GDSC,969,713,100.000000
1,PRISM HTS002,480,341,47.826087
2,PRISM MTS005,444,315,44.179523
3,PRISM MTS006,476,338,47.405330
4,PRISM MTS010,473,335,46.984572


In [11]:
# =============================================================================
# Assess deterministic CTRP-to-anchor name compatibility
# =============================================================================

with zipfile.ZipFile(
    pharmacogenomic_input_paths[("ctrp", "response_archive")]
) as ctrp_archive:
    ctrp_cell_lines = pd.read_csv(
        ctrp_archive.open("v20.meta.per_cell_line.txt"),
        sep="\t",
        usecols=["master_ccl_id", "ccl_name"],
    )

ctrp_cell_lines["name_key"] = (
    ctrp_cell_lines["ccl_name"]
    .astype(str)
    .str.upper()
    .str.replace(r"[^A-Z0-9]", "", regex=True)
)

anchor_name_map = (
    phase6_anchor_cohort[
        ["ModelID", "CellLineName"]
    ]
    .copy()
)

anchor_name_map["name_key"] = (
    anchor_name_map["CellLineName"]
    .astype(str)
    .str.upper()
    .str.replace(r"[^A-Z0-9]", "", regex=True)
)

ctrp_key_counts = ctrp_cell_lines["name_key"].value_counts()
anchor_key_counts = anchor_name_map["name_key"].value_counts()

shared_keys = set(ctrp_key_counts.index) & set(anchor_key_counts.index)

unambiguous_shared_keys = {
    key
    for key in shared_keys
    if ctrp_key_counts[key] == 1
    and anchor_key_counts[key] == 1
}

print("CTRP cell lines:", ctrp_cell_lines["master_ccl_id"].nunique())
print("Anchor models:", anchor_name_map["ModelID"].nunique())
print("Shared normalized name keys:", len(shared_keys))
print(
    "Unambiguous one-to-one shared keys:",
    len(unambiguous_shared_keys),
)
print(
    "Shared keys with CTRP collisions:",
    sum(ctrp_key_counts[key] > 1 for key in shared_keys),
)
print(
    "Shared keys with anchor collisions:",
    sum(anchor_key_counts[key] > 1 for key in shared_keys),
)

CTRP cell lines: 1107
Anchor models: 713
Shared normalized name keys: 623
Unambiguous one-to-one shared keys: 623
Shared keys with CTRP collisions: 0
Shared keys with anchor collisions: 0


In [12]:
# =============================================================================
# Quantify CTRP response-covered anchor models
# =============================================================================

ctrp_anchor_crosswalk = (
    ctrp_cell_lines.loc[
        ctrp_cell_lines["name_key"].isin(unambiguous_shared_keys),
        ["master_ccl_id", "ccl_name", "name_key"],
    ]
    .merge(
        anchor_name_map.loc[
            anchor_name_map["name_key"].isin(unambiguous_shared_keys),
            ["ModelID", "CellLineName", "name_key"],
        ],
        on="name_key",
        how="inner",
        validate="one_to_one",
    )
)

with zipfile.ZipFile(
    pharmacogenomic_input_paths[("ctrp", "response_archive")]
) as ctrp_archive:
    ctrp_response_experiments = pd.read_csv(
        ctrp_archive.open("v20.data.curves_post_qc.txt"),
        sep="\t",
        usecols=["experiment_id"],
    )
    ctrp_experiment_map = (
        pd.read_csv(
            ctrp_archive.open("v20.meta.per_experiment.txt"),
            sep="\t",
            usecols=["experiment_id", "master_ccl_id"],
        )
        .drop_duplicates()
    )

response_master_ccl_ids = set(
    ctrp_response_experiments
    .merge(
        ctrp_experiment_map,
        on="experiment_id",
        how="left",
        validate="many_to_one",
    )["master_ccl_id"]
    .dropna()
)

ctrp_response_crosswalk = ctrp_anchor_crosswalk.loc[
    ctrp_anchor_crosswalk["master_ccl_id"].isin(
        response_master_ccl_ids
    )
].copy()

print("Deterministic CTRP-anchor matches:", len(ctrp_anchor_crosswalk))
print(
    "Matches represented in post-QC response:",
    len(ctrp_response_crosswalk),
)
print(
    "Anchor models without deterministic CTRP response coverage:",
    len(anchor_model_ids - set(ctrp_response_crosswalk["ModelID"])),
)

Deterministic CTRP-anchor matches: 623
Matches represented in post-QC response: 566
Anchor models without deterministic CTRP response coverage: 147


In [13]:
# =============================================================================
# Characterize CTRP anchor-coverage loss
# =============================================================================

ctrp_matched_model_ids = set(ctrp_anchor_crosswalk["ModelID"])
ctrp_response_model_ids = set(ctrp_response_crosswalk["ModelID"])

ctrp_coverage_status = phase6_anchor_cohort[
    ["ModelID", "CellLineName", "OncotreeLineage"]
].copy()

ctrp_coverage_status["ctrp_name_match"] = (
    ctrp_coverage_status["ModelID"].isin(ctrp_matched_model_ids)
)
ctrp_coverage_status["ctrp_post_qc_response"] = (
    ctrp_coverage_status["ModelID"].isin(ctrp_response_model_ids)
)

ctrp_coverage_status["coverage_status"] = "no_deterministic_name_match"
ctrp_coverage_status.loc[
    ctrp_coverage_status["ctrp_name_match"],
    "coverage_status",
] = "matched_without_post_qc_response"
ctrp_coverage_status.loc[
    ctrp_coverage_status["ctrp_post_qc_response"],
    "coverage_status",
] = "response_covered"

print(
    ctrp_coverage_status["coverage_status"]
    .value_counts()
)

print(
    "\nCTRP anchor response coverage:",
    f"{100 * len(ctrp_response_model_ids) / len(anchor_model_ids):.2f}%",
)

coverage_status
response_covered                    566
no_deterministic_name_match          90
matched_without_post_qc_response     57
Name: count, dtype: int64

CTRP anchor response coverage: 79.38%


In [14]:
# =============================================================================
# Evaluate secondary deterministic CTRP model mapping
# =============================================================================

unmatched_anchor_ids = (
    anchor_model_ids - ctrp_matched_model_ids
)

depmap_stripped_names = pd.read_csv(
    Paths.depmap / "Model.csv",
    usecols=["ModelID", "StrippedCellLineName"],
)

depmap_stripped_names = (
    depmap_stripped_names.loc[
        depmap_stripped_names["ModelID"].isin(unmatched_anchor_ids)
    ]
    .dropna(subset=["StrippedCellLineName"])
    .copy()
)

depmap_stripped_names["stripped_name_key"] = (
    depmap_stripped_names["StrippedCellLineName"]
    .astype(str)
    .str.upper()
    .str.replace(r"[^A-Z0-9]", "", regex=True)
)

used_ctrp_ids = set(
    ctrp_anchor_crosswalk["master_ccl_id"]
)

available_ctrp_cell_lines = (
    ctrp_cell_lines.loc[
        ~ctrp_cell_lines["master_ccl_id"].isin(used_ctrp_ids)
    ]
    .copy()
)

available_ctrp_cell_lines["stripped_name_key"] = (
    available_ctrp_cell_lines["ccl_name"]
    .astype(str)
    .str.upper()
    .str.replace(r"[^A-Z0-9]", "", regex=True)
)

anchor_key_counts = (
    depmap_stripped_names["stripped_name_key"]
    .value_counts()
)
ctrp_key_counts = (
    available_ctrp_cell_lines["stripped_name_key"]
    .value_counts()
)

shared_stripped_keys = (
    set(anchor_key_counts.index)
    & set(ctrp_key_counts.index)
)

unambiguous_stripped_keys = {
    key
    for key in shared_stripped_keys
    if anchor_key_counts[key] == 1
    and ctrp_key_counts[key] == 1
}

print("Previously unmatched anchor models:", len(unmatched_anchor_ids))
print(
    "Models with available StrippedCellLineName:",
    depmap_stripped_names["ModelID"].nunique(),
)
print(
    "Shared stripped-name keys:",
    len(shared_stripped_keys),
)
print(
    "Unambiguous secondary matches:",
    len(unambiguous_stripped_keys),
)
print(
    "Ambiguous shared keys:",
    len(shared_stripped_keys - unambiguous_stripped_keys),
)

Previously unmatched anchor models: 90
Models with available StrippedCellLineName: 90
Shared stripped-name keys: 0
Unambiguous secondary matches: 0
Ambiguous shared keys: 0


In [15]:
# =============================================================================
# Construct anchor-restricted CTRP model crosswalk
# =============================================================================

ctrp_model_crosswalk = (
    ctrp_response_crosswalk[
        [
            "master_ccl_id",
            "ccl_name",
            "ModelID",
            "CellLineName",
        ]
    ]
    .assign(mapping_method="exact_normalized_cell_line_name")
    .sort_values("ModelID")
    .reset_index(drop=True)
)

print("Final CTRP mapped models:", len(ctrp_model_crosswalk))
print(
    "Unique ModelID:",
    ctrp_model_crosswalk["ModelID"].nunique(),
)
print(
    "Unique master_ccl_id:",
    ctrp_model_crosswalk["master_ccl_id"].nunique(),
)
print(
    "Anchor coverage:",
    f"{100 * len(ctrp_model_crosswalk) / len(anchor_model_ids):.2f}%",
)

Final CTRP mapped models: 566
Unique ModelID: 566
Unique master_ccl_id: 566
Anchor coverage: 79.38%


In [16]:
# =============================================================================
# Summarize cross-resource anchor-model coverage
# =============================================================================

prism_any_screen_ids = set(prism_models["depmap_id"])
prism_any_screen_overlap = (
    prism_any_screen_ids & anchor_model_ids
)

model_coverage_summary = pd.DataFrame(
    [
        {
            "resource": "GDSC",
            "anchor_models": len(anchor_model_ids),
            "response_covered_anchor_models": len(gdsc_anchor_overlap),
            "anchor_coverage_pct": (
                100 * len(gdsc_anchor_overlap) / len(anchor_model_ids)
            ),
        },
        {
            "resource": "CTRP",
            "anchor_models": len(anchor_model_ids),
            "response_covered_anchor_models": len(ctrp_model_crosswalk),
            "anchor_coverage_pct": (
                100 * len(ctrp_model_crosswalk) / len(anchor_model_ids)
            ),
        },
        {
            "resource": "PRISM any screen",
            "anchor_models": len(anchor_model_ids),
            "response_covered_anchor_models": len(prism_any_screen_overlap),
            "anchor_coverage_pct": (
                100 * len(prism_any_screen_overlap) / len(anchor_model_ids)
            ),
        },
    ]
)

model_coverage_summary

,resource,anchor_models,response_covered_anchor_models,anchor_coverage_pct
0,GDSC,713,713,100.000000
1,CTRP,713,566,79.382889
2,PRISM any screen,713,341,47.826087


In [17]:
# =============================================================================
# Reproduce frozen Phase 4 cell-line consensus scores
# =============================================================================

harmonized_expression = pd.read_parquet(
    upstream_artifact_paths["phase3.303.harmonized_expression"]
)

shared_gene_universe = pd.read_csv(
    upstream_artifact_paths[
        "phase4.400.cross_system_shared_gene_universe"
    ]
)

consensus_gene_weights = pd.read_csv(
    upstream_artifact_paths[
        "phase4.401.consensus_transcriptomic_gene_weights"
    ]
)

program_columns = [
    column
    for column in consensus_cellline_scores.columns
    if column != "ModelID"
]

consensus_weight_matrix = (
    consensus_gene_weights
    .pivot(
        index="gene_symbol",
        columns="consensus_program_id",
        values="consensus_weight",
    )
    .reindex(shared_gene_universe["gene_symbol"])
    [program_columns]
)

cell_line_projection_columns = (
    shared_gene_universe["cell_line_gene_id"].tolist()
)

expression_matrix = (
    harmonized_expression[cell_line_projection_columns]
    .to_numpy(dtype=np.float64)
)

gene_means = expression_matrix.mean(axis=0, keepdims=True)
gene_stds = expression_matrix.std(axis=0, ddof=1, keepdims=True)

standardized_expression = (
    expression_matrix - gene_means
) / gene_stds

projected_scores = (
    standardized_expression
    @ consensus_weight_matrix.to_numpy(dtype=np.float64)
)

score_means = projected_scores.mean(axis=0, keepdims=True)
score_stds = projected_scores.std(axis=0, ddof=1, keepdims=True)

reproduced_score_values = (
    projected_scores - score_means
) / score_stds

reproduced_scores = pd.DataFrame(
    reproduced_score_values,
    columns=program_columns,
)

reproduced_scores.insert(
    0,
    "ModelID",
    harmonized_expression["ModelID"].to_numpy(),
)

score_comparison = reproduced_scores.merge(
    consensus_cellline_scores,
    on="ModelID",
    how="inner",
    validate="one_to_one",
    suffixes=("_reproduced", "_frozen"),
)

max_absolute_differences = {
    program: (
        score_comparison[f"{program}_reproduced"]
        - score_comparison[f"{program}_frozen"]
    )
    .abs()
    .max()
    for program in program_columns
}

print("Models compared:", len(score_comparison))
print("Maximum absolute differences:")
for program, difference in max_absolute_differences.items():
    print(f"  {program}: {difference:.3e}")

Models compared: 713
Maximum absolute differences:
  CONSENSUS_TX_01: 2.665e-15
  CONSENSUS_TX_02: 1.776e-15
  CONSENSUS_TX_03: 2.220e-15


In [18]:
# =============================================================================
# Quantify potential model expansion beyond the frozen anchor cohort
# =============================================================================

ctrp_response_cell_ids = set(response_master_ccl_ids)
ctrp_anchor_response_cell_ids = set(
    ctrp_model_crosswalk["master_ccl_id"]
)

prism_response_model_ids = set(prism_models["depmap_id"])
prism_anchor_response_model_ids = (
    prism_response_model_ids & anchor_model_ids
)

potential_expansion_summary = pd.DataFrame(
    [
        {
            "resource": "CTRP",
            "response_models_or_cells": len(ctrp_response_cell_ids),
            "mapped_anchor_models": len(ctrp_anchor_response_cell_ids),
            "outside_anchor_or_unresolved": (
                len(ctrp_response_cell_ids)
                - len(ctrp_anchor_response_cell_ids)
            ),
        },
        {
            "resource": "PRISM",
            "response_models_or_cells": len(prism_response_model_ids),
            "mapped_anchor_models": len(prism_anchor_response_model_ids),
            "outside_anchor_or_unresolved": (
                len(prism_response_model_ids)
                - len(prism_anchor_response_model_ids)
            ),
        },
    ]
)

potential_expansion_summary

,resource,response_models_or_cells,mapped_anchor_models,outside_anchor_or_unresolved
0,CTRP,887,566,321
1,PRISM,480,341,139


In [19]:
# =============================================================================
# Quantify expression-eligible external models in CTRP and PRISM
# =============================================================================

depmap_models = pd.read_csv(
    Paths.depmap / "Model.csv",
    usecols=["ModelID", "CellLineName"],
)

depmap_models["name_key"] = (
    depmap_models["CellLineName"]
    .astype(str)
    .str.upper()
    .str.replace(r"[^A-Z0-9]", "", regex=True)
)

depmap_key_counts = depmap_models["name_key"].value_counts()
ctrp_key_counts_full = ctrp_cell_lines["name_key"].value_counts()

shared_full_keys = (
    set(depmap_key_counts.index)
    & set(ctrp_key_counts_full.index)
)

unambiguous_full_keys = {
    key
    for key in shared_full_keys
    if depmap_key_counts[key] == 1
    and ctrp_key_counts_full[key] == 1
}

ctrp_depmap_crosswalk = (
    ctrp_cell_lines.loc[
        ctrp_cell_lines["name_key"].isin(unambiguous_full_keys),
        ["master_ccl_id", "ccl_name", "name_key"],
    ]
    .merge(
        depmap_models.loc[
            depmap_models["name_key"].isin(unambiguous_full_keys),
            ["ModelID", "CellLineName", "name_key"],
        ],
        on="name_key",
        how="inner",
        validate="one_to_one",
    )
)

ctrp_external_model_ids = set(
    ctrp_depmap_crosswalk.loc[
        ctrp_depmap_crosswalk["master_ccl_id"].isin(
            response_master_ccl_ids
        ),
        "ModelID",
    ]
) - anchor_model_ids

prism_external_model_ids = (
    prism_response_model_ids - anchor_model_ids
)

depmap_expression_model_ids = set(
    pd.read_csv(
        Paths.depmap / "OmicsExpressionProteinCodingGenesTPMLogp1.csv",
        usecols=["Unnamed: 0"],
    )["Unnamed: 0"]
)

external_expression_coverage = pd.DataFrame(
    [
        {
            "resource": "CTRP",
            "deterministic_external_models": len(
                ctrp_external_model_ids
            ),
            "with_depmap_expression": len(
                ctrp_external_model_ids
                & depmap_expression_model_ids
            ),
        },
        {
            "resource": "PRISM",
            "deterministic_external_models": len(
                prism_external_model_ids
            ),
            "with_depmap_expression": len(
                prism_external_model_ids
                & depmap_expression_model_ids
            ),
        },
    ]
)

external_expression_coverage["expression_coverage_pct"] = (
    100
    * external_expression_coverage["with_depmap_expression"]
    / external_expression_coverage["deterministic_external_models"]
)

external_expression_coverage

,resource,deterministic_external_models,with_depmap_expression,expression_coverage_pct
0,CTRP,272,256,94.117647
1,PRISM,139,136,97.841727


In [20]:
# =============================================================================
# Quantify unique expression-eligible external model expansion
# =============================================================================

ctrp_external_expression_ids = (
    ctrp_external_model_ids
    & depmap_expression_model_ids
)

prism_external_expression_ids = (
    prism_external_model_ids
    & depmap_expression_model_ids
)

shared_external_expression_ids = (
    ctrp_external_expression_ids
    & prism_external_expression_ids
)

external_expression_union_ids = (
    ctrp_external_expression_ids
    | prism_external_expression_ids
)

print(
    "CTRP expression-eligible external models:",
    len(ctrp_external_expression_ids),
)
print(
    "PRISM expression-eligible external models:",
    len(prism_external_expression_ids),
)
print(
    "Shared external models across CTRP and PRISM:",
    len(shared_external_expression_ids),
)
print(
    "Unique external models across both screens:",
    len(external_expression_union_ids),
)
print(
    "Total scoreable models after expansion:",
    len(anchor_model_ids | external_expression_union_ids),
)
print(
    "Increase over frozen 713-model anchor:",
    f"{100 * len(external_expression_union_ids) / len(anchor_model_ids):.2f}%",
)

CTRP expression-eligible external models: 256
PRISM expression-eligible external models: 136
Shared external models across CTRP and PRISM: 124
Unique external models across both screens: 268
Total scoreable models after expansion: 981
Increase over frozen 713-model anchor: 37.59%


In [21]:
# =============================================================================
# Project consensus scores to expression-eligible external models
# =============================================================================

external_expression = pd.read_csv(
    Paths.depmap / "OmicsExpressionProteinCodingGenesTPMLogp1.csv",
    usecols=["Unnamed: 0", *cell_line_projection_columns],
).rename(columns={"Unnamed: 0": "ModelID"})

external_expression = (
    external_expression.loc[
        external_expression["ModelID"].isin(
            external_expression_union_ids
        )
    ]
    .copy()
)

external_expression_matrix = (
    external_expression[cell_line_projection_columns]
    .to_numpy(dtype=np.float64)
)

external_standardized_expression = (
    external_expression_matrix - gene_means
) / gene_stds

external_projected_scores = (
    external_standardized_expression
    @ consensus_weight_matrix.to_numpy(dtype=np.float64)
)

external_score_values = (
    external_projected_scores - score_means
) / score_stds

external_consensus_scores = pd.DataFrame(
    external_score_values,
    columns=program_columns,
)

external_consensus_scores.insert(
    0,
    "ModelID",
    external_expression["ModelID"].to_numpy(),
)

print(
    "Expected external models:",
    len(external_expression_union_ids),
)
print(
    "Projected external models:",
    len(external_consensus_scores),
)
print(
    "Missing expected models:",
    len(
        external_expression_union_ids
        - set(external_consensus_scores["ModelID"])
    ),
)
print(
    "Missing projected scores:",
    int(
        external_consensus_scores[
            program_columns
        ].isna().sum().sum()
    ),
)

Expected external models: 268
Projected external models: 268
Missing expected models: 0
Missing projected scores: 0


In [22]:
# =============================================================================
# Build expanded Phase 6 score universe with lineage metadata
# =============================================================================

external_model_metadata = pd.read_csv(
    Paths.depmap / "Model.csv",
    usecols=["ModelID", "OncotreeLineage"],
)

external_model_metadata = external_model_metadata.loc[
    external_model_metadata["ModelID"].isin(
        external_expression_union_ids
    )
].copy()

anchor_score_universe = (
    phase6_anchor_cohort[
        ["ModelID", "OncotreeLineage", *program_columns]
    ]
    .assign(score_origin="frozen_phase4")
)

external_score_universe = (
    external_consensus_scores
    .merge(
        external_model_metadata,
        on="ModelID",
        how="left",
        validate="one_to_one",
    )
    .assign(score_origin="projected_from_frozen_phase4")
)

phase6_score_universe = pd.concat(
    [
        anchor_score_universe,
        external_score_universe[
            ["ModelID", "OncotreeLineage", *program_columns, "score_origin"]
        ],
    ],
    ignore_index=True,
)

print("Phase 6 score universe:", len(phase6_score_universe))
print(
    "Unique ModelID:",
    phase6_score_universe["ModelID"].nunique(),
)
print(
    "Frozen Phase 4 models:",
    (phase6_score_universe["score_origin"] == "frozen_phase4").sum(),
)
print(
    "Projected external models:",
    (
        phase6_score_universe["score_origin"]
        == "projected_from_frozen_phase4"
    ).sum(),
)
print(
    "Models missing lineage:",
    phase6_score_universe["OncotreeLineage"].isna().sum(),
)
print(
    "External models missing lineage:",
    external_score_universe["OncotreeLineage"].isna().sum(),
)

Phase 6 score universe: 981
Unique ModelID: 981
Frozen Phase 4 models: 713
Projected external models: 268
Models missing lineage: 0
External models missing lineage: 0


In [23]:
# =============================================================================
# Characterize scoreable model coverage by resource and lineage
# =============================================================================

phase6_scoreable_ids = set(
    phase6_score_universe["ModelID"]
)

gdsc_scoreable_ids = set(
    phase6_anchor_cohort.loc[
        phase6_anchor_cohort["SangerModelID"].isin(
            gdsc_response_ids
        ),
        "ModelID",
    ]
)

ctrp_scoreable_ids = set(
    ctrp_depmap_crosswalk.loc[
        ctrp_depmap_crosswalk["master_ccl_id"].isin(
            response_master_ccl_ids
        ),
        "ModelID",
    ]
) & phase6_scoreable_ids

prism_scoreable_ids = (
    prism_response_model_ids
    & phase6_scoreable_ids
)

scoreable_resource_models = {
    "GDSC": gdsc_scoreable_ids,
    "CTRP": ctrp_scoreable_ids,
    "PRISM": prism_scoreable_ids,
}

resource_coverage_rows = []
lineage_coverage_rows = []

for resource, model_ids in scoreable_resource_models.items():
    resource_models = phase6_score_universe.loc[
        phase6_score_universe["ModelID"].isin(model_ids),
        ["ModelID", "OncotreeLineage"],
    ]

    resource_coverage_rows.append(
        {
            "resource": resource,
            "scoreable_models": len(resource_models),
            "represented_lineages": (
                resource_models["OncotreeLineage"].nunique()
            ),
        }
    )

    lineage_counts = (
        resource_models["OncotreeLineage"]
        .value_counts()
    )

    for lineage, count in lineage_counts.items():
        lineage_coverage_rows.append(
            {
                "resource": resource,
                "OncotreeLineage": lineage,
                "models": count,
            }
        )

resource_scoreable_coverage = pd.DataFrame(
    resource_coverage_rows
)

lineage_scoreable_coverage = (
    pd.DataFrame(lineage_coverage_rows)
    .pivot(
        index="OncotreeLineage",
        columns="resource",
        values="models",
    )
    .fillna(0)
    .astype(int)
)

display(resource_scoreable_coverage)
display(lineage_scoreable_coverage)

,resource,scoreable_models,represented_lineages
0,GDSC,713,27
1,CTRP,821,25
2,PRISM,477,22


resource,CTRP,GDSC,PRISM
OncotreeLineage,,,
Adrenal Gland,0,1,0
Ampulla of Vater,2,0,1
Biliary Tract,6,2,6
Bladder/Urinary Tract,24,16,23
Bone,15,17,12
Bowel,48,43,27
Breast,40,47,22
CNS/Brain,49,37,34
Cervix,1,11,0


In [24]:
# =============================================================================
# Reconcile CTRP anchor and expanded deterministic crosswalks
# =============================================================================

ctrp_anchor_response_ids = set(
    ctrp_model_crosswalk["ModelID"]
)

ctrp_full_response_ids = set(
    ctrp_depmap_crosswalk.loc[
        ctrp_depmap_crosswalk["master_ccl_id"].isin(
            response_master_ccl_ids
        ),
        "ModelID",
    ]
)

lost_anchor_ids = (
    ctrp_anchor_response_ids
    - ctrp_full_response_ids
)

external_full_ids = (
    ctrp_full_response_ids
    - anchor_model_ids
)

print(
    "Anchor CTRP models from original crosswalk:",
    len(ctrp_anchor_response_ids),
)
print(
    "Anchor CTRP models retained in full-DepMap crosswalk:",
    len(ctrp_full_response_ids & anchor_model_ids),
)
print(
    "Anchor models lost after full-DepMap ambiguity check:",
    len(lost_anchor_ids),
)
print(
    "External CTRP models in full-DepMap crosswalk:",
    len(external_full_ids),
)

if lost_anchor_ids:
    lost_keys = set(
        ctrp_anchor_crosswalk.loc[
            ctrp_anchor_crosswalk["ModelID"].isin(lost_anchor_ids),
            "name_key",
        ]
    )

    print("\nLost anchor mapping:")
    display(
        ctrp_anchor_crosswalk.loc[
            ctrp_anchor_crosswalk["ModelID"].isin(lost_anchor_ids),
            [
                "master_ccl_id",
                "ccl_name",
                "name_key",
                "ModelID",
                "CellLineName",
            ],
        ]
    )

    print("\nDepMap models sharing the affected normalized key:")
    display(
        depmap_models.loc[
            depmap_models["name_key"].isin(lost_keys),
            [
                "ModelID",
                "CellLineName",
                "name_key",
            ],
        ].sort_values(["name_key", "ModelID"])
    )

Anchor CTRP models from original crosswalk: 566
Anchor CTRP models retained in full-DepMap crosswalk: 565
Anchor models lost after full-DepMap ambiguity check: 1
External CTRP models in full-DepMap crosswalk: 272

Lost anchor mapping:


,master_ccl_id,ccl_name,name_key,ModelID,CellLineName
244,538,KMH2,KMH2,ACH-000815,KM-H2



DepMap models sharing the affected normalized key:


,ModelID,CellLineName,name_key
811,ACH-000815,KM-H2,KMH2
1868,ACH-002397,KMH-2,KMH2


In [25]:
# =============================================================================
# Finalize globally unambiguous CTRP Phase 6 model crosswalk
# =============================================================================

ctrp_phase6_crosswalk = (
    ctrp_depmap_crosswalk.loc[
        ctrp_depmap_crosswalk["master_ccl_id"].isin(
            response_master_ccl_ids
        )
        & ctrp_depmap_crosswalk["ModelID"].isin(
            phase6_scoreable_ids
        ),
        [
            "master_ccl_id",
            "ccl_name",
            "ModelID",
            "CellLineName",
        ],
    ]
    .assign(
        mapping_method="exact_normalized_name_globally_unambiguous"
    )
    .sort_values("ModelID")
    .reset_index(drop=True)
)

ctrp_phase6_anchor_ids = (
    set(ctrp_phase6_crosswalk["ModelID"])
    & anchor_model_ids
)

ctrp_phase6_external_ids = (
    set(ctrp_phase6_crosswalk["ModelID"])
    - anchor_model_ids
)

print(
    "Final CTRP Phase 6 scoreable models:",
    len(ctrp_phase6_crosswalk),
)
print(
    "Anchor models:",
    len(ctrp_phase6_anchor_ids),
)
print(
    "Projected external models:",
    len(ctrp_phase6_external_ids),
)
print(
    "Unique ModelID:",
    ctrp_phase6_crosswalk["ModelID"].nunique(),
)
print(
    "Unique master_ccl_id:",
    ctrp_phase6_crosswalk["master_ccl_id"].nunique(),
)
print(
    "Excluded globally ambiguous CTRP cell lines:",
    len(lost_anchor_ids),
)

Final CTRP Phase 6 scoreable models: 821
Anchor models: 565
Projected external models: 256
Unique ModelID: 821
Unique master_ccl_id: 821
Excluded globally ambiguous CTRP cell lines: 1


In [26]:
# =============================================================================
# Characterize native drug-level model coverage across pharmacogenomic resources
# =============================================================================

gdsc_drug_models = (
    pd.read_excel(
        pharmacogenomic_input_paths[("gdsc", "response")],
        usecols=["SANGER_MODEL_ID", "DRUG_ID"],
    )
    .merge(
        phase6_anchor_cohort[
            ["ModelID", "SangerModelID"]
        ],
        left_on="SANGER_MODEL_ID",
        right_on="SangerModelID",
        how="inner",
        validate="many_to_one",
    )
    [["DRUG_ID", "ModelID"]]
    .drop_duplicates()
)

with zipfile.ZipFile(
    pharmacogenomic_input_paths[("ctrp", "response_archive")]
) as ctrp_archive:
    ctrp_drug_models = pd.read_csv(
        ctrp_archive.open("v20.data.curves_post_qc.txt"),
        sep="\t",
        usecols=["experiment_id", "master_cpd_id"],
    )

ctrp_drug_models = (
    ctrp_drug_models
    .merge(
        ctrp_experiment_map,
        on="experiment_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        ctrp_phase6_crosswalk[
            ["master_ccl_id", "ModelID"]
        ],
        on="master_ccl_id",
        how="inner",
        validate="many_to_one",
    )
    [["master_cpd_id", "ModelID"]]
    .drop_duplicates()
)

prism_drug_models = (
    pd.read_csv(
        pharmacogenomic_input_paths[("prism", "response")],
        usecols=["broad_id", "depmap_id", "screen_id"],
    )
    .loc[
        lambda data: data["depmap_id"].isin(
            phase6_scoreable_ids
        )
    ]
    [["screen_id", "broad_id", "depmap_id"]]
    .drop_duplicates()
)

drug_coverage_rows = []

for resource, data, drug_column, model_column in (
    ("GDSC", gdsc_drug_models, "DRUG_ID", "ModelID"),
    ("CTRP", ctrp_drug_models, "master_cpd_id", "ModelID"),
):
    model_counts = data.groupby(drug_column)[model_column].nunique()

    drug_coverage_rows.append(
        {
            "resource": resource,
            "native_drugs": len(model_counts),
            "min_models_per_drug": int(model_counts.min()),
            "median_models_per_drug": float(model_counts.median()),
            "max_models_per_drug": int(model_counts.max()),
        }
    )

for screen_id, screen_data in prism_drug_models.groupby("screen_id"):
    model_counts = (
        screen_data.groupby("broad_id")["depmap_id"].nunique()
    )

    drug_coverage_rows.append(
        {
            "resource": f"PRISM {screen_id}",
            "native_drugs": len(model_counts),
            "min_models_per_drug": int(model_counts.min()),
            "median_models_per_drug": float(model_counts.median()),
            "max_models_per_drug": int(model_counts.max()),
        }
    )

native_drug_coverage = pd.DataFrame(drug_coverage_rows)

native_drug_coverage

,resource,native_drugs,min_models_per_drug,median_models_per_drug,max_models_per_drug
0,GDSC,295,136,656.0,713
1,CTRP,545,83,750.0,807
2,PRISM HTS002,1396,27,436.5,474
3,PRISM MTS005,2,407,408.0,409
4,PRISM MTS006,73,401,453.0,470
5,PRISM MTS010,147,358,430.0,460


In [27]:
# =============================================================================
# Construct native compound identity catalogs
# =============================================================================

gdsc_compounds = (
    pd.read_excel(
        pharmacogenomic_input_paths[("gdsc", "response")],
        usecols=["DRUG_ID", "DRUG_NAME"],
    )
    .drop_duplicates()
)

with zipfile.ZipFile(
    pharmacogenomic_input_paths[("ctrp", "response_archive")]
) as ctrp_archive:
    ctrp_compounds = pd.read_csv(
        ctrp_archive.open("v20.meta.per_compound.txt"),
        sep="\t",
        usecols=["master_cpd_id", "cpd_name"],
    )

prism_compounds = (
    pd.read_csv(
        pharmacogenomic_input_paths[("prism", "response")],
        usecols=["broad_id", "name"],
    )
    .drop_duplicates()
)

for data, name_column in (
    (gdsc_compounds, "DRUG_NAME"),
    (ctrp_compounds, "cpd_name"),
    (prism_compounds, "name"),
):
    data["drug_name_key"] = (
        data[name_column]
        .astype(str)
        .str.strip()
        .str.upper()
        .str.replace(r"[^A-Z0-9]", "", regex=True)
    )

compound_identity_summary = pd.DataFrame(
    [
        {
            "resource": "GDSC",
            "native_ids": gdsc_compounds["DRUG_ID"].nunique(),
            "native_names": gdsc_compounds["DRUG_NAME"].nunique(),
            "normalized_name_keys": gdsc_compounds["drug_name_key"].nunique(),
            "normalized_key_collisions": (
                gdsc_compounds.groupby("drug_name_key")["DRUG_ID"]
                .nunique()
                .gt(1)
                .sum()
            ),
        },
        {
            "resource": "CTRP",
            "native_ids": ctrp_compounds["master_cpd_id"].nunique(),
            "native_names": ctrp_compounds["cpd_name"].nunique(),
            "normalized_name_keys": ctrp_compounds["drug_name_key"].nunique(),
            "normalized_key_collisions": (
                ctrp_compounds.groupby("drug_name_key")["master_cpd_id"]
                .nunique()
                .gt(1)
                .sum()
            ),
        },
        {
            "resource": "PRISM",
            "native_ids": prism_compounds["broad_id"].nunique(),
            "native_names": prism_compounds["name"].nunique(),
            "normalized_name_keys": prism_compounds["drug_name_key"].nunique(),
            "normalized_key_collisions": (
                prism_compounds.groupby("drug_name_key")["broad_id"]
                .nunique()
                .gt(1)
                .sum()
            ),
        },
    ]
)

compound_identity_summary

,resource,native_ids,native_names,normalized_name_keys,normalized_key_collisions
0,GDSC,295,286,286,9
1,CTRP,545,545,545,0
2,PRISM,1502,1448,1446,56


In [28]:
# =============================================================================
# Quantify unambiguous exact-name compound overlap across resources
# =============================================================================

gdsc_unique_keys = {
    key
    for key, count in (
        gdsc_compounds.groupby("drug_name_key")["DRUG_ID"]
        .nunique()
        .items()
    )
    if count == 1
}

ctrp_unique_keys = {
    key
    for key, count in (
        ctrp_compounds.groupby("drug_name_key")["master_cpd_id"]
        .nunique()
        .items()
    )
    if count == 1
}

prism_unique_keys = {
    key
    for key, count in (
        prism_compounds.groupby("drug_name_key")["broad_id"]
        .nunique()
        .items()
    )
    if count == 1
}

gdsc_ctrp_keys = (
    gdsc_unique_keys
    & ctrp_unique_keys
)

gdsc_prism_keys = (
    gdsc_unique_keys
    & prism_unique_keys
)

ctrp_prism_keys = (
    ctrp_unique_keys
    & prism_unique_keys
)

three_way_keys = (
    gdsc_unique_keys
    & ctrp_unique_keys
    & prism_unique_keys
)

exact_name_overlap_summary = pd.DataFrame(
    [
        {
            "comparison": "GDSC–CTRP",
            "unambiguous_exact_name_matches": len(gdsc_ctrp_keys),
        },
        {
            "comparison": "GDSC–PRISM",
            "unambiguous_exact_name_matches": len(gdsc_prism_keys),
        },
        {
            "comparison": "CTRP–PRISM",
            "unambiguous_exact_name_matches": len(ctrp_prism_keys),
        },
        {
            "comparison": "GDSC–CTRP–PRISM",
            "unambiguous_exact_name_matches": len(three_way_keys),
        },
    ]
)

exact_name_overlap_summary

,comparison,unambiguous_exact_name_matches
0,GDSC–CTRP,68
1,GDSC–PRISM,87
2,CTRP–PRISM,151
3,GDSC–CTRP–PRISM,39


In [29]:
# =============================================================================
# Construct unambiguous exact-name cross-resource compound crosswalk
# =============================================================================

gdsc_exact = (
    gdsc_compounds.loc[
        gdsc_compounds["drug_name_key"].isin(gdsc_unique_keys),
        ["drug_name_key", "DRUG_ID", "DRUG_NAME"],
    ]
    .rename(
        columns={
            "DRUG_ID": "gdsc_drug_id",
            "DRUG_NAME": "gdsc_drug_name",
        }
    )
)

ctrp_exact = (
    ctrp_compounds.loc[
        ctrp_compounds["drug_name_key"].isin(ctrp_unique_keys),
        ["drug_name_key", "master_cpd_id", "cpd_name"],
    ]
    .rename(
        columns={
            "master_cpd_id": "ctrp_master_cpd_id",
            "cpd_name": "ctrp_drug_name",
        }
    )
)

prism_exact = (
    prism_compounds.loc[
        prism_compounds["drug_name_key"].isin(prism_unique_keys),
        ["drug_name_key", "broad_id", "name"],
    ]
    .rename(
        columns={
            "broad_id": "prism_broad_id",
            "name": "prism_drug_name",
        }
    )
)

exact_drug_crosswalk = (
    gdsc_exact
    .merge(
        ctrp_exact,
        on="drug_name_key",
        how="outer",
        validate="one_to_one",
    )
    .merge(
        prism_exact,
        on="drug_name_key",
        how="outer",
        validate="one_to_one",
    )
)

exact_drug_crosswalk["resources_present"] = (
    exact_drug_crosswalk[
        [
            "gdsc_drug_id",
            "ctrp_master_cpd_id",
            "prism_broad_id",
        ]
    ]
    .notna()
    .sum(axis=1)
)

cross_resource_exact_drugs = (
    exact_drug_crosswalk.loc[
        exact_drug_crosswalk["resources_present"] >= 2
    ]
    .sort_values(
        ["resources_present", "drug_name_key"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

print(
    "Exact-name compounds present in >=2 resources:",
    len(cross_resource_exact_drugs),
)
print(
    "Present in all 3 resources:",
    (cross_resource_exact_drugs["resources_present"] == 3).sum(),
)
print(
    "Present in exactly 2 resources:",
    (cross_resource_exact_drugs["resources_present"] == 2).sum(),
)

display(cross_resource_exact_drugs.head())

Exact-name compounds present in >=2 resources: 228
Present in all 3 resources: 39
Present in exactly 2 resources: 189


,drug_name_key,gdsc_drug_id,gdsc_drug_name,ctrp_master_cpd_id,ctrp_drug_name,prism_broad_id,prism_drug_name,resources_present
0,ABT737,1910.0,ABT737,411738.0,ABT-737,BRD-K56301217-001-07-4,ABT-737,3
1,ALISERTIB,1051.0,Alisertib,636711.0,alisertib,BRD-K75295174-001-05-0,alisertib,3
2,AXITINIB,1021.0,Axitinib,348990.0,axitinib,BRD-K29905972-001-06-3,axitinib,3
3,AZD4547,1786.0,AZD4547,660325.0,AZD4547,BRD-K28392481-001-05-1,AZD4547,3
4,AZD6482,2169.0,AZD6482,639390.0,AZD6482,BRD-K58772419-001-07-0,AZD6482,3


In [30]:
# =============================================================================
# Annotate exact cross-resource compounds with model coverage
# =============================================================================

gdsc_model_counts = (
    gdsc_drug_models
    .groupby("DRUG_ID")["ModelID"]
    .nunique()
    .rename("gdsc_models")
)

ctrp_model_counts = (
    ctrp_drug_models
    .groupby("master_cpd_id")["ModelID"]
    .nunique()
    .rename("ctrp_models")
)

prism_model_counts = (
    prism_drug_models
    .groupby(["broad_id", "screen_id"])["depmap_id"]
    .nunique()
    .unstack(fill_value=0)
    .rename(
        columns=lambda screen: f"prism_{screen}_models"
    )
)

cross_resource_drug_coverage = (
    cross_resource_exact_drugs
    .merge(
        gdsc_model_counts,
        left_on="gdsc_drug_id",
        right_index=True,
        how="left",
        validate="many_to_one",
    )
    .merge(
        ctrp_model_counts,
        left_on="ctrp_master_cpd_id",
        right_index=True,
        how="left",
        validate="many_to_one",
    )
    .merge(
        prism_model_counts,
        left_on="prism_broad_id",
        right_index=True,
        how="left",
        validate="many_to_one",
    )
)

coverage_columns = [
    column
    for column in cross_resource_drug_coverage.columns
    if column.endswith("_models")
]

cross_resource_drug_coverage[
    coverage_columns
] = (
    cross_resource_drug_coverage[
        coverage_columns
    ]
    .fillna(0)
    .astype(int)
)

print(
    "Cross-resource exact compounds:",
    len(cross_resource_drug_coverage),
)

print("\nPRISM screen availability among exact cross-resource compounds:")
for column in [
    column
    for column in coverage_columns
    if column.startswith("prism_")
]:
    print(
        f"  {column}:",
        (cross_resource_drug_coverage[column] > 0).sum(),
    )

display(
    cross_resource_drug_coverage[
        [
            "drug_name_key",
            "resources_present",
            *coverage_columns,
        ]
    ].head(10)
)

Cross-resource exact compounds: 228

PRISM screen availability among exact cross-resource compounds:
  prism_HTS002_models: 192
  prism_MTS005_models: 0
  prism_MTS006_models: 10
  prism_MTS010_models: 37


,drug_name_key,resources_present,gdsc_models,ctrp_models,prism_HTS002_models,prism_MTS005_models,prism_MTS006_models,prism_MTS010_models
0,ABT737,3,703,708,452,0,0,0
1,ALISERTIB,3,697,774,471,0,0,0
2,AXITINIB,3,706,773,437,0,0,430
3,AZD4547,3,706,702,344,0,0,0
4,AZD6482,3,175,785,473,0,0,0
5,AZD7762,3,707,788,466,0,0,0
6,AZD8055,3,579,794,450,0,0,0
7,BI2536,3,675,790,465,0,0,0
8,BIBR1532,3,701,773,467,0,0,0
9,BMS345541,3,658,754,453,0,0,0


In [31]:
# =============================================================================
# Characterize PRISM screen patterns among cross-resource compounds
# =============================================================================

prism_screen_columns = [
    "prism_HTS002_models",
    "prism_MTS005_models",
    "prism_MTS006_models",
    "prism_MTS010_models",
]

prism_cross_resource_compounds = (
    cross_resource_drug_coverage.loc[
        cross_resource_drug_coverage["prism_broad_id"].notna()
    ]
    .copy()
)

prism_cross_resource_compounds["screen_pattern"] = (
    prism_cross_resource_compounds[
        prism_screen_columns
    ]
    .gt(0)
    .apply(
        lambda row: "+".join(
            column.removeprefix("prism_").removesuffix("_models")
            for column, present in row.items()
            if present
        ),
        axis=1,
    )
)

prism_screen_pattern_summary = (
    prism_cross_resource_compounds["screen_pattern"]
    .value_counts()
    .rename_axis("screen_pattern")
    .reset_index(name="compounds")
)

display(prism_screen_pattern_summary)

print(
    "Cross-resource compounds represented in PRISM:",
    len(prism_cross_resource_compounds),
)
print(
    "Compounds with MTS010:",
    (
        prism_cross_resource_compounds["prism_MTS010_models"] > 0
    ).sum(),
)
print(
    "MTS010 compounds also represented in HTS002:",
    (
        (prism_cross_resource_compounds["prism_MTS010_models"] > 0)
        & (prism_cross_resource_compounds["prism_HTS002_models"] > 0)
    ).sum(),
)
print(
    "MTS006 compounds also represented in HTS002:",
    (
        (prism_cross_resource_compounds["prism_MTS006_models"] > 0)
        & (prism_cross_resource_compounds["prism_HTS002_models"] > 0)
    ).sum(),
)

,screen_pattern,compounds
0,HTS002,156
1,HTS002+MTS010,33
2,MTS006,4
3,MTS006+MTS010,3
4,HTS002+MTS006,2
5,HTS002+MTS006+MTS010,1


Cross-resource compounds represented in PRISM: 199
Compounds with MTS010: 37
MTS010 compounds also represented in HTS002: 34
MTS006 compounds also represented in HTS002: 3


In [32]:
# =============================================================================
# Define prospective primary PRISM screen rule
# =============================================================================

def assign_primary_prism_screen(row):
    if row["prism_MTS010_models"] > 0:
        return "MTS010"

    available_nonredo_screens = [
        screen
        for screen in ("HTS002", "MTS005", "MTS006")
        if row[f"prism_{screen}_models"] > 0
    ]

    if len(available_nonredo_screens) == 1:
        return available_nonredo_screens[0]

    if len(available_nonredo_screens) > 1:
        return "AMBIGUOUS_NONREDO"

    return pd.NA


prism_cross_resource_compounds["primary_prism_screen"] = (
    prism_cross_resource_compounds.apply(
        assign_primary_prism_screen,
        axis=1,
    )
)

primary_prism_screen_summary = (
    prism_cross_resource_compounds["primary_prism_screen"]
    .value_counts(dropna=False)
    .rename_axis("primary_prism_screen")
    .reset_index(name="compounds")
)

display(primary_prism_screen_summary)

print(
    "Compounds eligible for a unique primary PRISM screen:",
    (
        prism_cross_resource_compounds["primary_prism_screen"]
        != "AMBIGUOUS_NONREDO"
    ).sum(),
)
print(
    "Compounds retained for screen-specific sensitivity only:",
    (
        prism_cross_resource_compounds["primary_prism_screen"]
        == "AMBIGUOUS_NONREDO"
    ).sum(),
)

,primary_prism_screen,compounds
0,HTS002,156
1,MTS010,37
2,MTS006,4
3,AMBIGUOUS_NONREDO,2


Compounds eligible for a unique primary PRISM screen: 197
Compounds retained for screen-specific sensitivity only: 2


In [33]:
print(primary_prism_screen_summary)

print(
    "Compounds eligible for a unique primary PRISM screen:",
    (
        prism_cross_resource_compounds["primary_prism_screen"]
        != "AMBIGUOUS_NONREDO"
    ).sum(),
)
print(
    "Compounds retained for screen-specific sensitivity only:",
    (
        prism_cross_resource_compounds["primary_prism_screen"]
        == "AMBIGUOUS_NONREDO"
    ).sum(),
)

  primary_prism_screen  compounds
0               HTS002        156
1               MTS010         37
2               MTS006          4
3    AMBIGUOUS_NONREDO          2
Compounds eligible for a unique primary PRISM screen: 197
Compounds retained for screen-specific sensitivity only: 2


In [34]:
# =============================================================================
# Define prospective primary pharmacogenomic response metrics
# =============================================================================

PRIMARY_RESPONSE_METRICS = {
    "GDSC": {
        "metric": "LN_IC50",
        "higher_is": "resistance_like",
        "role": "developmental_internal",
    },
    "CTRP": {
        "metric": "area_under_curve",
        "higher_is": "resistance_like",
        "role": "external_replication",
    },
    "PRISM": {
        "metric": "auc",
        "higher_is": "resistance_like",
        "role": "external_replication",
    },
}

primary_response_metric_summary = pd.DataFrame(
    [
        {
            "resource": resource,
            **specification,
        }
        for resource, specification
        in PRIMARY_RESPONSE_METRICS.items()
    ]
)

primary_response_metric_summary

,resource,metric,higher_is,role
0,GDSC,LN_IC50,resistance_like,developmental_internal
1,CTRP,area_under_curve,resistance_like,external_replication
2,PRISM,auc,resistance_like,external_replication


In [35]:
# =============================================================================
# Characterize repeated CTRP AUC measurements in the Phase 6 universe
# =============================================================================

with zipfile.ZipFile(
    pharmacogenomic_input_paths[("ctrp", "response_archive")]
) as ctrp_archive:
    ctrp_auc_response = pd.read_csv(
        ctrp_archive.open("v20.data.curves_post_qc.txt"),
        sep="\t",
        usecols=[
            "experiment_id",
            "master_cpd_id",
            "area_under_curve",
        ],
    )

ctrp_auc_phase6 = (
    ctrp_auc_response
    .merge(
        ctrp_experiment_map,
        on="experiment_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        ctrp_phase6_crosswalk[
            ["master_ccl_id", "ModelID"]
        ],
        on="master_ccl_id",
        how="inner",
        validate="many_to_one",
    )
)

ctrp_pair_summary = (
    ctrp_auc_phase6
    .groupby(["ModelID", "master_cpd_id"])["area_under_curve"]
    .agg(
        experiments="size",
        auc_min="min",
        auc_max="max",
    )
    .reset_index()
)

repeated_ctrp_pairs = (
    ctrp_pair_summary.loc[
        ctrp_pair_summary["experiments"] > 1
    ]
    .copy()
)

repeated_ctrp_pairs["auc_range"] = (
    repeated_ctrp_pairs["auc_max"]
    - repeated_ctrp_pairs["auc_min"]
)

print(
    "Phase 6 CTRP model-compound pairs:",
    len(ctrp_pair_summary),
)
print(
    "Pairs with repeated experiments:",
    len(repeated_ctrp_pairs),
)
print(
    "Repeated-pair percentage:",
    f"{100 * len(repeated_ctrp_pairs) / len(ctrp_pair_summary):.2f}%",
)
print(
    "Maximum experiments per pair:",
    int(ctrp_pair_summary["experiments"].max()),
)

print("\nAUC range among repeated pairs:")
print(
    repeated_ctrp_pairs["auc_range"]
    .quantile([0.50, 0.75, 0.90, 0.95, 0.99, 1.00])
)

Phase 6 CTRP model-compound pairs: 357460
Pairs with repeated experiments: 7708
Repeated-pair percentage: 2.16%
Maximum experiments per pair: 3

AUC range among repeated pairs:
0.50     0.765500
0.75     1.521500
0.90     2.550270
0.95     3.393020
0.99     5.698351
1.00    11.500000
Name: auc_range, dtype: float64


In [36]:
# =============================================================================
# Construct primary aggregated CTRP Phase 6 response
# =============================================================================

ctrp_primary_response = (
    ctrp_auc_phase6
    .groupby(
        ["ModelID", "master_cpd_id"],
        as_index=False,
    )
    .agg(
        response_value=(
            "area_under_curve",
            "median",
        ),
        n_experiments=(
            "experiment_id",
            "nunique",
        ),
    )
)

ctrp_primary_response["response_metric"] = (
    "area_under_curve"
)

ctrp_primary_response["aggregation_rule"] = (
    "median_across_experiments"
)

print(
    "Primary CTRP model-compound observations:",
    len(ctrp_primary_response),
)
print(
    "Unique ModelID-compound pairs:",
    ctrp_primary_response[
        ["ModelID", "master_cpd_id"]
    ]
    .drop_duplicates()
    .shape[0],
)
print(
    "Pairs based on one experiment:",
    (ctrp_primary_response["n_experiments"] == 1).sum(),
)
print(
    "Pairs based on repeated experiments:",
    (ctrp_primary_response["n_experiments"] > 1).sum(),
)
print(
    "Missing primary response values:",
    ctrp_primary_response["response_value"]
    .isna()
    .sum(),
)

Primary CTRP model-compound observations: 357460
Unique ModelID-compound pairs: 357460
Pairs based on one experiment: 349752
Pairs based on repeated experiments: 7708
Missing primary response values: 0


In [37]:
# =============================================================================
# Evaluate candidate drug estimability and lineage-support rule
# =============================================================================

MIN_MODELS_PER_LINEAGE = 20
MIN_SUPPORTED_LINEAGES = 3
MIN_SUPPORTED_MODELS = 100


def summarize_drug_eligibility(
    drug_model_pairs,
    drug_id_column,
    resource,
):
    pairs_with_lineage = (
        drug_model_pairs[
            [drug_id_column, "ModelID"]
        ]
        .drop_duplicates()
        .merge(
            phase6_score_universe[
                ["ModelID", "OncotreeLineage"]
            ],
            on="ModelID",
            how="inner",
            validate="many_to_one",
        )
    )

    lineage_counts = (
        pairs_with_lineage
        .groupby(
            [drug_id_column, "OncotreeLineage"]
        )["ModelID"]
        .nunique()
        .rename("models")
        .reset_index()
    )

    supported_lineages = (
        lineage_counts.loc[
            lineage_counts["models"]
            >= MIN_MODELS_PER_LINEAGE
        ]
        .copy()
    )

    eligibility = (
        supported_lineages
        .groupby(drug_id_column)
        .agg(
            supported_lineages=(
                "OncotreeLineage",
                "nunique",
            ),
            supported_models=(
                "models",
                "sum",
            ),
        )
        .reindex(
            drug_model_pairs[
                drug_id_column
            ].drop_duplicates()
        )
        .fillna(0)
        .reset_index()
    )

    eligibility[
        ["supported_lineages", "supported_models"]
    ] = eligibility[
        ["supported_lineages", "supported_models"]
    ].astype(int)

    eligibility["eligible"] = (
        (
            eligibility["supported_lineages"]
            >= MIN_SUPPORTED_LINEAGES
        )
        & (
            eligibility["supported_models"]
            >= MIN_SUPPORTED_MODELS
        )
    )

    eligibility["resource"] = resource

    return eligibility


gdsc_eligibility = summarize_drug_eligibility(
    gdsc_drug_models[
        ["DRUG_ID", "ModelID"]
    ],
    drug_id_column="DRUG_ID",
    resource="GDSC",
)

ctrp_eligibility = summarize_drug_eligibility(
    ctrp_primary_response[
        ["master_cpd_id", "ModelID"]
    ],
    drug_id_column="master_cpd_id",
    resource="CTRP",
)

prism_primary_assignment = (
    prism_cross_resource_compounds.loc[
        prism_cross_resource_compounds[
            "primary_prism_screen"
        ].notna()
        & (
            prism_cross_resource_compounds[
                "primary_prism_screen"
            ]
            != "AMBIGUOUS_NONREDO"
        ),
        [
            "prism_broad_id",
            "primary_prism_screen",
        ],
    ]
    .drop_duplicates()
)

prism_primary_pairs = (
    prism_drug_models
    .rename(
        columns={
            "depmap_id": "ModelID",
        }
    )
    .merge(
        prism_primary_assignment,
        left_on="broad_id",
        right_on="prism_broad_id",
        how="inner",
        validate="many_to_one",
    )
)

prism_primary_pairs = (
    prism_primary_pairs.loc[
        prism_primary_pairs["screen_id"]
        == prism_primary_pairs[
            "primary_prism_screen"
        ],
        ["broad_id", "ModelID"],
    ]
    .drop_duplicates()
)

prism_eligibility = summarize_drug_eligibility(
    prism_primary_pairs,
    drug_id_column="broad_id",
    resource="PRISM_exact_cross_resource",
)

eligibility_summary = pd.DataFrame(
    [
        {
            "resource": "GDSC",
            "drugs_evaluated": len(gdsc_eligibility),
            "drugs_eligible": gdsc_eligibility["eligible"].sum(),
        },
        {
            "resource": "CTRP",
            "drugs_evaluated": len(ctrp_eligibility),
            "drugs_eligible": ctrp_eligibility["eligible"].sum(),
        },
        {
            "resource": "PRISM_exact_cross_resource",
            "drugs_evaluated": len(prism_eligibility),
            "drugs_eligible": prism_eligibility["eligible"].sum(),
        },
    ]
)

eligibility_summary["eligible_fraction"] = (
    eligibility_summary["drugs_eligible"]
    / eligibility_summary["drugs_evaluated"]
)

display(eligibility_summary)

,resource,drugs_evaluated,drugs_eligible,eligible_fraction
0,GDSC,295,281,0.952542
1,CTRP,545,499,0.915596
2,PRISM_exact_cross_resource,197,194,0.984772


### Primary drug estimability rule

An outcome-blind coverage assessment was used to evaluate the prespecified
candidate eligibility rule before any program–drug association results were
inspected.

The primary Phase 6 rule is frozen as:

- at least 20 response-covered models per supported lineage;
- at least 3 supported lineages; and
- at least 100 models in total across supported lineages.

Only supported lineages contribute to the primary drug-specific model.

This rule retains 281/295 GDSC drugs, 499/545 CTRP compounds, and 194/197
PRISM exact cross-resource compounds with a unique primary screen assignment.

The authoritative methodological definition and rationale are recorded in
`docs/contracts/phase6/PHASE6_ANALYSIS_CONTRACT.md`.

In [38]:
# =============================================================================
# Construct primary PRISM Phase 6 response
# =============================================================================

prism_response = pd.read_csv(
    pharmacogenomic_input_paths[("prism", "response")],
    usecols=[
        "broad_id",
        "depmap_id",
        "screen_id",
        "auc",
    ],
)

prism_primary_response = (
    prism_response
    .rename(
        columns={
            "depmap_id": "ModelID",
            "auc": "response_value",
        }
    )
    .loc[
        lambda data: data["ModelID"].isin(
            phase6_scoreable_ids
        )
    ]
    .merge(
        prism_primary_assignment,
        left_on="broad_id",
        right_on="prism_broad_id",
        how="inner",
        validate="many_to_one",
    )
    .loc[
        lambda data:
        data["screen_id"]
        == data["primary_prism_screen"]
    ]
    .drop(
        columns="prism_broad_id"
    )
    .reset_index(drop=True)
)

prism_primary_response["response_metric"] = "auc"

prism_pair_duplicates = (
    prism_primary_response
    .duplicated(
        subset=["ModelID", "broad_id"],
        keep=False,
    )
)

print(
    "Primary PRISM response rows:",
    len(prism_primary_response),
)
print(
    "Unique ModelID-compound pairs:",
    prism_primary_response[
        ["ModelID", "broad_id"]
    ]
    .drop_duplicates()
    .shape[0],
)
print(
    "Duplicated ModelID-compound pairs:",
    prism_pair_duplicates.sum(),
)
print(
    "Missing primary response values:",
    prism_primary_response[
        "response_value"
    ]
    .isna()
    .sum(),
)

print("\nPrimary PRISM observations by selected screen:")
print(
    prism_primary_response["screen_id"]
    .value_counts()
    .sort_index()
)

Primary PRISM response rows: 84802
Unique ModelID-compound pairs: 84802
Duplicated ModelID-compound pairs: 0
Missing primary response values: 0

Primary PRISM observations by selected screen:
screen_id
HTS002    67078
MTS006     1818
MTS010    15906
Name: count, dtype: int64


In [39]:
# =============================================================================
# Construct frozen lineage-supported pharmacogenomic analysis universes
# =============================================================================

def restrict_to_supported_drug_lineages(
    response_data,
    eligibility_data,
    drug_id_column,
    resource,
):
    eligible_drugs = set(
        eligibility_data.loc[
            eligibility_data["eligible"],
            drug_id_column,
        ]
    )

    response_with_lineage = (
        response_data
        .loc[
            lambda data:
            data[drug_id_column].isin(
                eligible_drugs
            )
        ]
        .merge(
            phase6_score_universe[
                ["ModelID", "OncotreeLineage"]
            ],
            on="ModelID",
            how="inner",
            validate="many_to_one",
        )
    )

    supported_drug_lineages = (
        response_with_lineage
        .groupby(
            [drug_id_column, "OncotreeLineage"]
        )["ModelID"]
        .nunique()
        .rename("models")
        .reset_index()
        .loc[
            lambda data:
            data["models"]
            >= MIN_MODELS_PER_LINEAGE
        ]
    )

    primary_universe = (
        response_with_lineage
        .merge(
            supported_drug_lineages[
                [drug_id_column, "OncotreeLineage"]
            ],
            on=[
                drug_id_column,
                "OncotreeLineage",
            ],
            how="inner",
            validate="many_to_one",
        )
        .reset_index(drop=True)
    )

    summary = {
        "resource": resource,
        "eligible_drugs": (
            primary_universe[
                drug_id_column
            ].nunique()
        ),
        "primary_observations": len(
            primary_universe
        ),
        "unique_models": (
            primary_universe["ModelID"]
            .nunique()
        ),
        "represented_lineages": (
            primary_universe[
                "OncotreeLineage"
            ].nunique()
        ),
    }

    return (
        primary_universe,
        supported_drug_lineages,
        summary,
    )


gdsc_primary_universe, gdsc_supported_lineages, gdsc_primary_summary = (
    restrict_to_supported_drug_lineages(
        gdsc_drug_models[
            ["DRUG_ID", "ModelID"]
        ],
        gdsc_eligibility,
        drug_id_column="DRUG_ID",
        resource="GDSC",
    )
)

ctrp_primary_universe, ctrp_supported_lineages, ctrp_primary_summary = (
    restrict_to_supported_drug_lineages(
        ctrp_primary_response,
        ctrp_eligibility,
        drug_id_column="master_cpd_id",
        resource="CTRP",
    )
)

prism_primary_universe, prism_supported_lineages, prism_primary_summary = (
    restrict_to_supported_drug_lineages(
        prism_primary_response,
        prism_eligibility,
        drug_id_column="broad_id",
        resource="PRISM_exact_cross_resource",
    )
)

primary_universe_summary = pd.DataFrame(
    [
        gdsc_primary_summary,
        ctrp_primary_summary,
        prism_primary_summary,
    ]
)

display(primary_universe_summary)

,resource,eligible_drugs,primary_observations,unique_models,represented_lineages
0,GDSC,281,136176,557,11
1,CTRP,499,302310,736,15
2,PRISM_exact_cross_resource,194,65736,386,11


In [40]:
# =============================================================================
# Load minimal GDSC response columns required for the Phase 6 primary response
# =============================================================================

gdsc_response = pd.read_excel(
    pharmacogenomic_input_paths[("gdsc", "response")],
    usecols=[
        "SANGER_MODEL_ID",
        "DRUG_ID",
        "LN_IC50",
    ],
)

In [41]:
# =============================================================================
# Construct primary GDSC Phase 6 response from the frozen model crosswalk
# =============================================================================

gdsc_model_crosswalk = (
    phase6_anchor_cohort[
        ["SangerModelID", "ModelID"]
    ]
    .rename(
        columns={
            "SangerModelID": "SANGER_MODEL_ID",
        }
    )
)

gdsc_primary_response = (
    gdsc_response
    .merge(
        gdsc_model_crosswalk,
        on="SANGER_MODEL_ID",
        how="inner",
        validate="many_to_one",
    )
    .rename(
        columns={
            "LN_IC50": "response_value",
        }
    )
    [
        [
            "ModelID",
            "DRUG_ID",
            "response_value",
        ]
    ]
)

gdsc_primary_response["response_metric"] = "LN_IC50"

gdsc_primary_universe = (
    gdsc_primary_universe[
        [
            "DRUG_ID",
            "ModelID",
            "OncotreeLineage",
        ]
    ]
    .merge(
        gdsc_primary_response,
        on=["DRUG_ID", "ModelID"],
        how="left",
        validate="one_to_one",
    )
)

print(
    "Primary GDSC response pairs:",
    len(gdsc_primary_response),
)
print(
    "Duplicated ModelID-drug pairs:",
    gdsc_primary_response
    .duplicated(
        subset=["ModelID", "DRUG_ID"]
    )
    .sum(),
)
print(
    "Missing primary response values:",
    gdsc_primary_response[
        "response_value"
    ]
    .isna()
    .sum(),
)

print(
    "\nLineage-supported primary GDSC observations:",
    len(gdsc_primary_universe),
)
print(
    "Eligible drugs:",
    gdsc_primary_universe[
        "DRUG_ID"
    ]
    .nunique(),
)
print(
    "Missing response values after support restriction:",
    gdsc_primary_universe[
        "response_value"
    ]
    .isna()
    .sum(),
)

Primary GDSC response pairs: 180614
Duplicated ModelID-drug pairs: 0
Missing primary response values: 0

Lineage-supported primary GDSC observations: 136176
Eligible drugs: 281
Missing response values after support restriction: 0


In [42]:
# =============================================================================
# Assemble the primary GDSC program–drug analysis frame
# =============================================================================

PROGRAM_COLUMNS = [
    "CONSENSUS_TX_01",
    "CONSENSUS_TX_02",
    "CONSENSUS_TX_03",
]

gdsc_primary_analysis = (
    gdsc_primary_universe
    .merge(
        phase6_score_universe[
            ["ModelID", *PROGRAM_COLUMNS]
        ],
        on="ModelID",
        how="left",
        validate="many_to_one",
    )
)

print(
    "Primary GDSC analysis observations:",
    len(gdsc_primary_analysis),
)
print(
    "Eligible drugs:",
    gdsc_primary_analysis["DRUG_ID"].nunique(),
)
print(
    "Unique models:",
    gdsc_primary_analysis["ModelID"].nunique(),
)
print(
    "Supported lineages:",
    gdsc_primary_analysis["OncotreeLineage"].nunique(),
)

print("\nMissing values:")
print(
    gdsc_primary_analysis[
        [
            "response_value",
            "OncotreeLineage",
            *PROGRAM_COLUMNS,
        ]
    ]
    .isna()
    .sum()
)

Primary GDSC analysis observations: 136176
Eligible drugs: 281
Unique models: 557
Supported lineages: 11

Missing values:
response_value     0
OncotreeLineage    0
CONSENSUS_TX_01    0
CONSENSUS_TX_02    0
CONSENSUS_TX_03    0
dtype: int64


In [43]:
# =============================================================================
# Verify algebraic estimability of the candidate primary association model
# =============================================================================

design_checks = []

for drug_id, drug_data in gdsc_primary_analysis.groupby(
    "DRUG_ID",
    sort=False,
):
    lineage_dummies = pd.get_dummies(
        drug_data["OncotreeLineage"],
        drop_first=True,
        dtype=float,
    )

    base_design = np.column_stack(
        [
            np.ones(len(drug_data)),
            lineage_dummies.to_numpy(),
        ]
    )

    response_has_variation = (
        drug_data["response_value"].nunique()
        > 1
    )

    for program in PROGRAM_COLUMNS:
        program_values = (
            drug_data[program]
            .to_numpy(dtype=float)
        )

        design_matrix = np.column_stack(
            [
                base_design,
                program_values,
            ]
        )

        design_rank = np.linalg.matrix_rank(
            design_matrix
        )

        design_checks.append(
            {
                "DRUG_ID": drug_id,
                "program": program,
                "n_models": len(drug_data),
                "n_lineages": (
                    drug_data[
                        "OncotreeLineage"
                    ].nunique()
                ),
                "response_has_variation": (
                    response_has_variation
                ),
                "program_has_variation": (
                    np.ptp(program_values) > 0
                ),
                "design_columns": (
                    design_matrix.shape[1]
                ),
                "design_rank": design_rank,
                "full_rank": (
                    design_rank
                    == design_matrix.shape[1]
                ),
            }
        )

design_checks = pd.DataFrame(
    design_checks
)

print(
    "Candidate drug-program models:",
    len(design_checks),
)
print(
    "Full-rank design matrices:",
    design_checks["full_rank"].sum(),
)
print(
    "Non-full-rank design matrices:",
    (~design_checks["full_rank"]).sum(),
)
print(
    "Models with constant response:",
    (~design_checks["response_has_variation"]).sum(),
)
print(
    "Models with constant program score:",
    (~design_checks["program_has_variation"]).sum(),
)

display(
    design_checks.loc[
        (~design_checks["full_rank"])
        | (~design_checks["response_has_variation"])
        | (~design_checks["program_has_variation"])
    ]
)

Candidate drug-program models: 843
Full-rank design matrices: 843
Non-full-rank design matrices: 0
Models with constant response: 0
Models with constant program score: 0


,DRUG_ID,program,n_models,n_lineages,response_has_variation,program_has_variation,design_columns,design_rank,full_rank


### Frozen primary association specification

Before inspection of any program–drug association result, the notebook-600
primary inferential family was frozen as 281 eligible GDSC drugs × 3 frozen
consensus programs = 843 tests.

Each association is estimated as:

`LN_IC50 ~ program_score + C(OncotreeLineage)`

using OLS with HC3 robust standard errors and only drug-specific supported
lineages under the frozen `20 / 3 / 100` rule.

Raw GDSC `LN_IC50` and the frozen Phase 4 score scale are retained without
drug-specific re-standardization.

Benjamini-Hochberg correction is applied jointly across all 843 primary tests,
with `q < 0.05` defining the FDR-controlled developmental association set.

CTRP and PRISM are not used in this primary family and remain reserved for
cross-screen replication.

Leave-one-lineage-out analyses are prespecified sensitivity characterizations
and do not alter primary inference.